In [ ]:
import numpy as np
from mod3d import BRepBuilderAPI, Render, gp
from pythreejs import (
    BufferGeometry, BufferAttribute, Mesh, Scene, 
    PerspectiveCamera, Renderer, AmbientLight, DirectionalLight,
    MeshBasicMaterial, OrbitControls, EdgesGeometry, LineSegments,
    LineBasicMaterial, MeshPhongMaterial, LineMaterial, Geometry, Line, MeshPhysicalMaterial,
    CylinderGeometry, ConeGeometry, SphereGeometry
)

## Helper Function to Convert OCCT Mesh to pythreejs

In [ ]:
def faces_mesh(faces_data):
    """
    Convert OCCT face tessellation data to a pythreejs mesh.
    
    Parameters:
    -----------
    faces_data : list
        List of tuples (triangles, vertices, normals, uvs) for each face
        
    Returns:
    --------
    mesh : pythreejs.Mesh or None
        The combined mesh for all faces with polygon offset to prevent z-fighting with edges
    """
    if not faces_data:
        return None
    
    # Combine all faces into single mesh
    all_vertices = []
    all_indices = []
    all_normals = []
    vertex_offset = 0
    
    for triangles, vertices, normals, uvs in faces_data:
        # Add vertices
        all_vertices.append(vertices)
        
        # Add normals if available
        if normals is not None:
            all_normals.append(normals)
        
        # Add indices with offset
        all_indices.append(triangles + vertex_offset)
        vertex_offset += vertices.shape[0]
    
    # Concatenate all data
    vertices_combined = np.vstack(all_vertices).astype(np.float32)
    indices_combined = np.vstack(all_indices).astype(np.uint32).ravel()
    
    # Create BufferGeometry
    geometry = BufferGeometry(
        attributes={
            'position': BufferAttribute(vertices_combined, normalized=False),
            'index': BufferAttribute(indices_combined, normalized=False)
        }
    )
    
    # Compute vertex normals for smooth shading
    geometry.exec_three_obj_method('computeVertexNormals')
    
    # Create material with polygon offset to prevent z-fighting with edges
    # polygonOffset pushes the faces back slightly in depth buffer
    material = MeshPhongMaterial(
        color='#2194ce',
        shininess=100,
        side='DoubleSide',
        polygonOffset=True,        # Enable polygon offset
        polygonOffsetFactor=1,     # Offset factor
        polygonOffsetUnits=1       # Offset units
    )

    material = MeshBasicMaterial(
        color='#2194ce',
        side='DoubleSide',
        polygonOffset=True,        # Enable polygon offset
        polygonOffsetFactor=1,     # Offset factor
        polygonOffsetUnits=1       # Offset units
    )

    material = MeshPhysicalMaterial(
        color='#049ef4',              # Bright blue color
        metalness=0.1,                # Low metalness for non-metallic appearance (0-1)
        roughness=1.0,                # Some roughness for realistic surface (0=mirror, 1=diffuse)
        clearcoat=0.5,                # Clear coating layer on top (0-1)
        clearcoatRoughness=0.1,       # Smooth clear coat
        reflectivity=0.5,             # How reflective the surface is (0-1)
        ior=1.5,                      # Index of refraction (glass=1.5, diamond=2.4)
        side='DoubleSide',            # Render both sides of faces
        polygonOffset=True,           # Enable polygon offset to prevent z-fighting with edges
        polygonOffsetFactor=1,        # Offset factor
        polygonOffsetUnits=1          # Offset units
    )
    
    mesh = Mesh(geometry=geometry, material=material)
    mesh.renderOrder = 0  # Render faces first
    return mesh

def edges_mesh(edges_data):
    """
    Convert OCCT edge tessellation data to pythreejs lines.
    
    Parameters:
    -----------
    edges_data : list
        List of tuples (indices, vertices) for each edge
        
    Returns:
    --------
    edge_lines : list or None
        List of pythreejs.Line objects, one per edge, with renderOrder set to appear on top
    """
    if not edges_data:
        return None
    
    # Create separate Line object for each edge
    edge_lines = []
    material = LineBasicMaterial(linewidth=5, color='#000000')
    
    for _, vertices in edges_data:
        # The vertices are already in the correct order for the polyline
        # Just use them directly to create continuous lines
        vertices_array = vertices.astype(np.float32)
        
        # Create geometry with vertices for continuous line
        geometry = Geometry(vertices=vertices_array.tolist())
        
        # Create continuous line for this edge
        line = Line(geometry=geometry, material=material)
        line.renderOrder = 1  # Render edges after faces to ensure visibility
        edge_lines.append(line)
    
    # Return list of edge lines
    return edge_lines

def occt_to_threejs(shape, linear_deflection=0.1, **kwargs):
    """
    Convert an OCCT shape to pythreejs mesh and edge lines.
    
    Parameters:
    -----------
    shape : TopoDS_Shape
        The OCCT shape to convert
    linear_deflection : float, optional
        The maximum distance between the triangulation and the surface (default: 0.1)
        Smaller values create finer meshes
    **kwargs : dict
        Additional arguments passed to extract_tessellation:
        - angle_deflection: Angular deflection in degrees (default: 20.0)
        - relative: Use relative deflection (default: False)
    
    Returns:
    --------
    mesh_face : pythreejs.Mesh
        The rendered mesh for faces
    mesh_edges : list of pythreejs.Line
        List of line objects for edges, rendered on top of faces
        
    Notes:
    ------
    - Faces use polygon offset to prevent z-fighting with edges
    - Edges are rendered as separate continuous polylines per edge
    - RenderOrder ensures edges appear on top of faces
    """
    # Extract tessellation data
    faces_data, edges_data = Render.extract_tessellation(shape, linear_deflection, **kwargs)
    
    mesh_face = faces_mesh(faces_data)
    mesh_edges = edges_mesh(edges_data)

    return mesh_face, mesh_edges

In [ ]:
class ShapeRenderer:
    """Simple renderer that can accumulate OCCT shapes and display them in one scene."""

    def __init__(self, linear_deflection=0.1, angle_deflection=15.0, width=1200, height=600):
        self.linear_deflection = linear_deflection
        self.angle_deflection = angle_deflection
        self.width = width
        self.height = height
        self._models = []

    def add_shape(self, shape):
        """Queue an OCCT shape for rendering."""
        self._models.append(shape)
        return shape

    def clear(self):
        """Drop all queued shapes."""
        self._models.clear()

    def _prepare_meshes(self):
        meshes = []
        for shape in self._models:
            mesh_face, mesh_edges = occt_to_threejs(
                shape,
                linear_deflection=self.linear_deflection,
                angle_deflection=self.angle_deflection,
            )
            meshes.append((mesh_face, mesh_edges))
        return meshes

    def render(self, background=None, lights=None):
        """Return a renderer showing everything that has been queued via `add_shape`."""
        if not self._models:
            raise RuntimeError("No shapes have been queued for rendering")

        camera = PerspectiveCamera(position=[0, 20, 40], aspect=self.width / self.height)
        default_lights = [
            AmbientLight(intensity=0.5),
            DirectionalLight(position=[10, 10, 10], intensity=0.5),
        ]
        lights = lights or default_lights

        scene_children = [camera, *lights]
        for mesh_face, mesh_edges in self._prepare_meshes():
            scene_children.append(mesh_face)
            if mesh_edges:
                scene_children.extend(mesh_edges)

        scene_kwargs = {"children": scene_children}
        if background is not None:
            scene_kwargs["background"] = background
        scene = Scene(**scene_kwargs)
        controls = OrbitControls(controlling=camera)
        return Renderer(
            camera=camera,
            scene=scene,
            controls=[controls],
            width=self.width,
            height=self.height,
            antialias=True,
        )


## Example 1: Simple Box

In [ ]:
# Create a box
box = BRepBuilderAPI.MakeBox(10.0, 20.0, 30.0).shape()

# Convert to threejs mesh
mesh_face, mesh_edges = occt_to_threejs(box, linear_deflection=0.5)

camera = PerspectiveCamera(position=[50, 50, 50], aspect=1.5)
# Setup scene (mesh_edges is now a list of lines)
scene_children = [mesh_face, camera, 
                  AmbientLight(intensity=0.5),
                  DirectionalLight(position=[100, 100, 100], intensity=0.5)]
if mesh_edges:
    scene_children.extend(mesh_edges)

scene = Scene(children=scene_children)

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)

from mod3d import TopExp
print(len(TopExp.get_vertices(box)))

renderer

## Example 2: Cylinder with Edges

In [ ]:
# Create a cylinder
cylinder = BRepBuilderAPI.MakeCylinder(5.0, 15.0).shape()

# Convert to mesh
mesh_face, _ = occt_to_threejs(cylinder, linear_deflection=0.2)

# Add edge visualization
edges = EdgesGeometry(mesh_face.geometry)
edge_lines = LineSegments(
    geometry=edges,
    material=LineBasicMaterial(color='black', linewidth=1)
)

# Setup scene
camera = PerspectiveCamera(position=[30, 30, 30], aspect=1.5)
scene = Scene(children=[mesh_face, edge_lines, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Example 3: Sphere

In [ ]:
# Create a sphere
sphere = BRepBuilderAPI.MakeSphere(8.0).shape()

# Convert to mesh with finer deflection
mesh_face, _  = occt_to_threejs(sphere, linear_deflection=0.1, angle_deflection=15.0)

# Setup scene
camera = PerspectiveCamera(position=[25, 25, 25], aspect=8.0/6.0)
scene = Scene(children=[mesh_face, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Example 4: Complex Shape - Filleted Box

In [ ]:
# Create a box with fillets
from mod3d import BRepFillet, TopExp

box = BRepBuilderAPI.MakeBox(20.0, 20.0, 20.0).shape()

# Create fillet
fillet_maker = BRepFillet.MakeFillet(box)

# Add fillets to edges
for edge in TopExp.get_edges(box):
    fillet_maker.add(3.0, edge)  # 3mm radius fillet

filleted_box = fillet_maker.shape()

# Convert to mesh
mesh_face, mesh_edges = occt_to_threejs(filleted_box, linear_deflection=0.2, angle_deflection=10.0)

# Setup scene
camera = PerspectiveCamera(position=[50, 50, 50], aspect=1.5)
scene_children = [mesh_face, camera,
                  AmbientLight(intensity=0.5),
                  DirectionalLight(position=[10, 10, 10], intensity=0.5)]
if mesh_edges:
    scene_children.extend(mesh_edges)

scene = Scene(children=scene_children)

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=800, height=600)
renderer

## Mesh Quality Comparison

Compare different tessellation quality settings

In [ ]:
sphere = BRepBuilderAPI.MakeSphere(5.0).shape()

# Coarse mesh
mesh_coarse, _ = occt_to_threejs(sphere, linear_deflection=1.0, angle_deflection=30.0)
mesh_coarse.position = [-15, 0, 0]

# Medium mesh
mesh_medium, _ = occt_to_threejs(sphere, linear_deflection=0.3, angle_deflection=15.0)
mesh_medium.position = [0, 0, 0]

# Fine mesh
mesh_fine, _ = occt_to_threejs(sphere, linear_deflection=0.1, angle_deflection=5.0)
mesh_fine.position = [15, 0, 0]

# Setup scene with all three meshes
camera = PerspectiveCamera(position=[0, 20, 40], aspect=2.0)
scene = Scene(children=[mesh_coarse, mesh_medium, mesh_fine, camera,
                        AmbientLight(intensity=0.5),
                        DirectionalLight(position=[10, 10, 10], intensity=0.5)])

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=1200, height=600)
renderer

In [ ]:
import mod3d

sphere = mod3d.BRepBuilderAPI.MakeSphere(5.0).shape()
box = mod3d.BRepBuilderAPI.MakeBox(10.0, 10.0, 10.0).shape()

# Move box to center on sphere
translation = mod3d.gp.Trsf()
translation.set_translation(mod3d.gp.Vec(-10.0, -10.0, -10.0))
box = box.moved(translation)

common = mod3d.BooleanOp.Common(sphere, box)
result = common.shape()

mesh_face, mesh_edges = occt_to_threejs(result, linear_deflection=0.1, angle_deflection=5.0)

# Setup scene
scene_children = [mesh_face, 
                  AmbientLight(intensity=0.5),
                  DirectionalLight(position=[10, 10, 10], intensity=0.5)]
if mesh_edges:
    scene_children.extend(mesh_edges)

camera = PerspectiveCamera(position=[0, 20, 40], aspect=2.0)
scene_children.append(camera)
scene = Scene(children=scene_children, background='lightgray')

controls = OrbitControls(controlling=camera)
renderer = Renderer(camera=camera, scene=scene, controls=[controls], width=1200, height=600, antialias=True,)
renderer

# Example 5: Visualize Point Cloud

In [ ]:
# simple test to display numpy array as points in 3D
import numpy as np

from pythreejs import (
    AmbientLight,
    BufferAttribute,
    BufferGeometry,
    DirectionalLight,
    OrbitControls,
    Points,
    PointsMaterial,
    PerspectiveCamera,
    Renderer,
    Scene,
)

vertices = np.array([[0, 0, 0], [10, 0, 0], [10, 10, 0], [10, 10, 10]], dtype=np.float32)
geometry = BufferGeometry(
    attributes={
        "position": BufferAttribute(vertices, normalized=False),
    }
)
material = PointsMaterial(color="blue", size=5.0, sizeAttenuation=False)
points_mesh = Points(geometry=geometry, material=material)

camera = PerspectiveCamera(position=[0, 20, 40], aspect=1.5)
default_lights = [
    AmbientLight(intensity=0.6),
    DirectionalLight(position=[10, 10, 10], intensity=0.6),
]

scene = Scene(children=[points_mesh, camera, *default_lights], background="lightgray")
controls = OrbitControls(controlling=camera)

renderer = Renderer(
    camera=camera,
    scene=scene,
    controls=[controls],
    width=1200,
    height=800,
    antialias=True,
)
renderer

In [ ]:
# simple test to display numpy array as flat spheres in 3D
import numpy as np

from pythreejs import (
    AmbientLight,
    BufferAttribute,
    BufferGeometry,
    DirectionalLight,
    OrbitControls,
    Points,
    PointsMaterial,
    PerspectiveCamera,
    Renderer,
    Scene,
)

vertices = np.array([[0, 0, 0], [10, 0, 0], [10, 10, 0], [10, 10, 10]], dtype=np.float32)
geometry = BufferGeometry(
    attributes={
        "position": BufferAttribute(vertices, normalized=False),
    }
)
material = PointsMaterial(color="blue", size=5.0, sizeAttenuation=False)
points_mesh = Points(geometry=geometry, material=material)

camera = PerspectiveCamera(position=[0, 20, 40], aspect=1.5)
default_lights = [
    AmbientLight(intensity=0.6),
    DirectionalLight(position=[10, 10, 10], intensity=0.6),
]

scene = Scene(children=[points_mesh, camera, *default_lights], background="lightgray")
controls = OrbitControls(controlling=camera)

renderer = Renderer(
    camera=camera,
    scene=scene,
    controls=[controls],
    width=1200,
    height=800,
    antialias=True,
)
renderer

In [ ]:
import pythreejs as p3
import numpy as np
from ipywidgets import HBox, VBox, Layout, Checkbox

def create_trihedron(size=1):
    """Create XYZ axes (trihedron) for orientation reference."""
    axes = []
    colors = ['red', 'green', 'blue']  # X=red, Y=green, Z=blue
    directions = [
        [[0, 0, 0], [size, 0, 0]],  # X
        [[0, 0, 0], [0, size, 0]],  # Y
        [[0, 0, 0], [0, 0, size]],  # Z
    ]

    for color, pts in zip(colors, directions):
        vertices = np.array(pts, dtype=np.float32)
        geometry = p3.BufferGeometry(attributes={
            'position': p3.BufferAttribute(vertices, normalized=False)
        })
        material = p3.LineBasicMaterial(color=color, linewidth=2)
        line = p3.Line(geometry=geometry, material=material)
        axes.append(line)

    return axes

def axis_angle_quaternion(axis, angle):
    axis = np.array(axis, dtype=np.float64)
    axis = axis / np.linalg.norm(axis) if np.linalg.norm(axis) > 0 else axis
    half_angle = angle / 2.0
    sin_half = np.sin(half_angle)
    cos_half = np.cos(half_angle)
    return (axis[0] * sin_half, axis[1] * sin_half, axis[2] * sin_half, cos_half)

def make_axis_decoration(color, axis, angle, cylinder_pos, cone_pos):
    quaternion = axis_angle_quaternion(axis, angle)
    cylinder = p3.Mesh(
        geometry=p3.CylinderGeometry(radiusTop=0.05, radiusBottom=0.05, height=0.9, radialSegments=16),
        material=p3.MeshBasicMaterial(color=color),
        quaternion=quaternion,
        position=cylinder_pos,
    )
    cone = p3.Mesh(
        geometry=p3.ConeGeometry(radius=0.1, height=0.3, radialSegments=16),
        material=p3.MeshBasicMaterial(color=color),
        quaternion=quaternion,
        position=cone_pos,
    )
    return cylinder, cone

# Round points using ShaderMaterial
N = 200
positions = (np.random.rand(N, 3) * 100 - 50).astype(np.float32)
geometry = p3.BufferGeometry(attributes={
    'position': p3.BufferAttribute(positions, normalized=False),
})

vertex_shader = """
uniform float size;
void main() {
    vec4 mvPosition = modelViewMatrix * vec4(position, 1.0);
    gl_PointSize = size;
    gl_Position = projectionMatrix * mvPosition;
}
"""

fragment_shader = """
uniform vec3 color;
void main() {
    vec2 center = gl_PointCoord - vec2(0.5);
    float dist = length(center);
    if (dist > 0.5) discard;
    gl_FragColor = vec4(color, 1.0);
}
"""

material = p3.ShaderMaterial(
    vertexShader=vertex_shader,
    fragmentShader=fragment_shader,
    uniforms={
        'size': {'value': 5.0},
        'color': {'value': [1.0, 0.0, 0.0]},
    },
    transparent=True,
    depthTest=True,
    depthWrite=False,
    blending='NormalBlending',
    )

points = p3.Points(geometry=geometry, material=material)

# Create trihedron as a Group so we can scale it uniformly

trihedron_axes = create_trihedron(size=1)
trihedron_lines = p3.Group(children=trihedron_axes)

axis_configs = [
    ('x', '#ff3333', [0, 0, 1], -np.pi / 2, [0.45, 0, 0], [0.95, 0, 0]),
    ('y', '#33ff33', [0, 0, 1], 0, [0, 0.45, 0], [0, 0.95, 0]),
    ('z', '#3333ff', [1, 0, 0], np.pi / 2, [0, 0, 0.45], [0, 0, 0.95]),
]
decorations = []
for _, color, axis, angle, cyl_pos, cone_pos in axis_configs:
    cylinder, cone = make_axis_decoration(color, axis, angle, cyl_pos, cone_pos)
    decorations.extend([cylinder, cone])
trihedron_decorations = p3.Group(children=decorations)

origin_sphere = p3.Mesh(
    geometry=p3.SphereGeometry(radius=0.08, widthSegments=16, heightSegments=16),
    material=p3.MeshBasicMaterial(
        color='gray',
        transparent=True,
        shininess=40,
    ),
    position=[0, 0, 0],
    )

helper_root = p3.Group(children=[trihedron_lines, trihedron_decorations, origin_sphere])

# Initial scale factor (will be updated based on camera distance)
initial_distance = np.sqrt(0**2 + 100**2 + 200**2)  # matches camera position
desired_screen_size = 20  # how big trihedron should appear
helper_root.scale = (desired_screen_size, desired_screen_size, desired_screen_size)
renderer_width = 1200
renderer_height = 800
grid_size = 1600
grid_helper = p3.GridHelper(size=grid_size, divisions=grid_size//10, colorCenterLine='#777777', colorGrid='#bbbbbb')
fog_near_base = 30
fog_far_base = grid_size / 2
scene_fog = p3.Fog(color='lightgray', near=fog_near_base, far=fog_far_base)

main_camera = p3.PerspectiveCamera(position=[0, 100, 200], fov=60, aspect=renderer_width / renderer_height)
main_scene = p3.Scene(
    children=[points, grid_helper, helper_root, p3.AmbientLight(color='#777777')],
    background='lightgray',
    fog=scene_fog
    )
main_controls = p3.OrbitControls(controlling=main_camera)
renderer = p3.Renderer(
    camera=main_camera, scene=main_scene, controls=[main_controls],
    width=renderer_width, height=renderer_height, antialias=True
    )

# Update trihedron scale and fog when camera moves
def update_trihedron_scale(change):
    pos = np.array(main_camera.position)
    distance = np.linalg.norm(pos)
    scale = distance * desired_screen_size / initial_distance
    helper_root.scale = (scale, scale, scale)
    scene_fog.near = max(fog_near_base, distance * 0.15)
    scene_fog.far = max(fog_far_base, grid_size / 2 + distance)

main_camera.observe(update_trihedron_scale, names=['position'])
update_trihedron_scale(None)

renderer